# The cost of ammonia production

Analysis of the cost of ammonia based on different technology pathways.

The source of nitrogen is likely to be air separation through the course of the energy transition.

However, the hydrogen production is likely to involve carbon capture and utilization (CCU) along side tradition methane reforming production [Blue pathway]
Or electrolysis [Green pathway].

To this end, the framework is used to analyze the cost of hydrogen production as a result of the technologies avaible at our disposal, as also the trade-offs interms of emissions when considering them.


Specific features of the study are as follows:

- H2 Use for Ammonia production:  400,000 to 500,000 kg H2/day; we may add other production volumes later

- H2 carbon intensity:  100% renewable electrolysis (0 kg CO2 / kg H2); 95% CCS + ATR/SMR (0.4 kg CO2/kg H2 – this qualifies for the $3/kg H2 PTC); There might be a 90% CCS case in some of the projects, but not sure yet on that. 

- H2 Cost:  The partners have provided no data, so we have to make some assumptions.  Let’s start with $1/kg H2 for now.  Or, maybe you can back out what this number should be based on the Ammonia market. 

- Ammonia price:  I’ll let you define this.  I think the goal would be for clean hydrogen based ammonia to be competitive with dirty hydrogen ammonia. 

__author__ = "Rahul Kakodkar", "Swaminathan Sundar", "Efstratios Pistikopoulos"
__copyright__ = "Copyright 2023, Multi-parametric Optimization & Control Lab"
__credits__ = ["Rahul Kakodkar", "Swaminathan Sundar", "Efstratios N. Pistikopoulos"]
__license__ = "Open"
__version__ = "1.0.0"
__maintainer__ = "Rahul Kakodkar"
__email__ = "cacodcar@tamu.edu"
__status__ = "Production"

In [31]:
import pandas 
from numpy import poly1d, polyfit, arange
from src.energiapy.components.temporal_scale import Temporal_scale
from src.energiapy.components.resource import Resource
from src.energiapy.components.process import Process
from src.energiapy.components.material import Material
from src.energiapy.components.location import Location
from src.energiapy.components.network import Network
from src.energiapy.components.scenario import Scenario
from src.energiapy.components.transport import Transport
from src.energiapy.components.result import Result 
from src.energiapy.model.formulate import formulate, Constraints, Objective
from src.energiapy.plot import plot
from src.energiapy.model.solve import solve
import matplotlib.pyplot as plt
from matplotlib import rc
from itertools import product
from pyomo.environ import Constraint, Set
from src.energiapy.utils.data_utils import get_data, make_henry_price_df



In [32]:
weather20_df = pandas.read_csv('data/ho_solar20.csv', index_col=0)
weather20_df.index = [i.split('+')[0] for i in weather20_df.index]
weather = weather20_df[~weather20_df.index.str.contains('02-29')] #remove leap years
weather

,wind_speed,dni
2020-01-01 00:00:00,9.5,0.0
2020-01-01 01:00:00,7.5,0.0
2020-01-01 02:00:00,6.0,0.0
2020-01-01 03:00:00,6.0,0.0
2020-01-01 04:00:00,6.0,0.0
...,...,...
2020-12-31 19:00:00,54.5,0.5
2020-12-31 20:00:00,55.5,0.0
2020-12-31 21:00:00,50.0,0.0
2020-12-31 22:00:00,46.0,181.5


In [33]:
#Actual temporal scale (daily)
ng_price20 = make_henry_price_df(
    file_name='data/Henry_Hub_Natural_Gas_Spot_Price_Daily.csv', year=2020, stretch=False)
ng_price_df = ng_price20
ng_price_df['index'] = [i for i in weather.index][::24]
ng_price_df = ng_price_df.drop(columns= 'scales')
ng_price = ng_price_df.set_index('index')
ng_price

,CH4
index,
2020-01-01 00:00:00,0.093304
2020-01-02 00:00:00,0.091518
2020-01-03 00:00:00,0.091964
2020-01-04 00:00:00,0.091964
2020-01-05 00:00:00,0.091964
...,...
2020-12-27 00:00:00,0.119643
2020-12-28 00:00:00,0.119643
2020-12-29 00:00:00,0.106696


**Constants**

In [34]:
bigM = 10**4  # very large number
smallM = 0.1
water_price = 31.70  # $/5000gallons
power_price = 8  # cents/kWh
ur_price = 42.70  # 250 Pfund U308 (Uranium)
A_f = 0.05  # annualization factor
# CO2_res = 0.2
pv_start = 0
ake_start = 0
smrh_start = 0
smr_start = 0
asmr_start = 0

In [35]:
cost_dict = get_data(file_name='data/cost_dict')
for i in cost_dict['HO']['moderate'].keys():
    print(i + ':', cost_dict['HO']['moderate'][i]['0'])

LiI_c: {'CAPEX': 1302181.81818181, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 0, 'units': '$/MW', 'source': 'NREL Annual Technology Baseline 2021, https://atb.nrel.gov/'}
LiI_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
CAES_c: {'CAPEX': 1669000.0, 'Fixed O&M': 16700.0, 'Variable O&M': 0, 'units': '$/MW', 'source': 'https://www.pnm.com/documents/396023/1506047/2017+-+HDR+10-30-17+PNM+Energy+Storage+Report.pdf/a2b7ca65-e1ba-92c8-308a-9a8391a87331'}
CAES_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
PSH_c: {'CAPEX': 0, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 4435.188, 'units': '$/MW', 'source': 'NREL Annual Technology Baseline 2021, https://atb.nrel.gov/'}
PSH_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
PV: {'CAPEX': 1302181.81818181, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 0, 'units': '$/MW', 'sour

**Resources**

In [36]:
Charge = Resource(name='Charge', store_max=100, basis='MW', label='Battery energy', block='energystorage')

Air_C = Resource(name='Air_C', store_max=bigM, basis='MW', label='CAES energy', block='energystorage')

H2O_E = Resource(name='H2O_E', store_max=bigM, basis='MW',
                 label='PSH energy', block='energystorage')

Uranium = Resource(name='Uranium', cons_max=(1/4.17*10**(-5))*bigM,
                   price=ur_price/(250/2), basis='kg', label='Uranium', block='energyfeedstock')

Solar = Resource(
    name='Solar', cons_max=bigM, basis='MW', label='Solar Power', block='energyfeedstock')

Wind = Resource(name='Wind', cons_max= bigM, basis='MW', label='Wind Power', block='energyfeedstock')

# H2 = Resource(name='H2', basis='kg', sell = True, demand = True, label='Hydrogen', block='Resource')
H2 = Resource(name='H2', basis='kg', label='Hydrogen', block='Resource')


H2O = Resource(name='H2O', cons_max=10**10,
               price=water_price/(5000*3.7854), basis='kg', label='Water', block='Resource')
            
O2 = Resource(name='O2', sell=True, loss=0.07,
              basis='kg', label='Oxygen', block='Resource')


CH4 = Resource(name='CH4', cons_max=100, price=0.113891, basis='kg', varying= True,\
    label='Natural gas', block='materialfeedstock')

CO2 = Resource(name='CO2', basis='kg',
               label='Carbon dioxide', block='Resource')

CO2_AQoff = Resource(
    name='CO2_AQoff', store_max=10**6, basis='kg', label='Carbon dioxide - sequestered', block='carbonsequestration')

CO2_EOR = Resource(
    name='CO2_EOR', store_max=10**6, basis='kg', label='Carbon dioxide - EOR', block='carbonsequestration')


CO2_Vent = Resource(
    name='CO2_Vent', sell=True, basis='kg', label='Carbon dioxide - Vented', block='resourcedischarge')

# Power= Resource(name= 'Power', sell= True, store_max=0,   \
#    mile= (10**3)/(0.2167432**1.60934), label= 'Renewable power generated')

Power = Resource(name='Power', basis='MW',
                 label='Renewable power generated', block='Resource')

#TODO

N2 = Resource(name='N2', basis='kg', label='Nitrogen', block='Resource')

NH3 = Resource(name='NH3', basis='kg', sell= True, demand= True, label='Ammonia', block='Resource')

Air = Resource(name='Air', cons_max = bigM, basis = 'MW', label='Air', block='materialfeedstock')



**Process**

In [37]:
LiI_c = Process(name='LiI_c', conversion={Charge: 1, Power: -1}, cost = cost_dict['HO']['moderate']['LiI_c']['0'],\
    prod_max=100, trl='nrel', block='power_storage', label='Lithium-ion battery', citation='Zakeri 2015')

LiI_d = Process(name='LiI_d', conversion={Charge: -1.1765, Power: 1}, cost =  {'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': smallM, \
    'units': '$/kg','source': 'dummy'}, \
    prod_max=100, trl='discharge', block='power_storage', label='Lithium-ion battery discharge', citation='Zakeri 2015')

CAES_c = Process(name='CAES_c', conversion={Air_C: 1, Power: -1}, cost = cost_dict['HO']['moderate']['CAES_c']['0'], \
    intro_scale=0, prod_max=bigM, trl='pilot', block='power_storage', label='Compressed air energy storage (CAES)', citation='Zakeri 2015')

CAES_d = Process(name='CAES_d', conversion={Air_C: -1.4286, Power: 1}, cost =  {'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': smallM, \
    'units': '$/kg','source': 'dummy'},\
    intro_scale=0, prod_max=bigM, trl='discharge', block='power_storage', label='Compressed air energy storage (CAES) discharge', citation='Zakeri 2015')

PSH_c = Process(name='PSH_c', conversion={H2O_E: 1, Power: -1}, cost = cost_dict['HO']['moderate']['PSH_c']['0'], \
    intro_scale=0, prod_max=bigM, trl='nrel', block='power_storage', label='Pumped storage hydropower (PSH)', citation='Zakeri 2015')

PSH_d = Process(name='PSH_d', conversion={H2O_E: -1.4286, Power: 1}, cost =  {'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': smallM, \
    'units': '$/kg','source': 'dummy'}, \
    prod_max=bigM, trl='discharge', block='power_storage', label='Pumped storage hydropower (PSH) discharge', citation='Zakeri 2015')

WF = Process(name='WF', conversion={Wind: -1, Power: 1, H2O: -1}, cost=cost_dict['HO']['moderate']['WF']['0'],
             prod_max=100, gwp=52700, land=10800/1800, trl='nrel', block='power_generation',
             label='Wind mill array', citation='Use windtoolkit conversion', varying= True)

PV = Process(name='PV', intro_scale=pv_start, conversion={Solar: -1, Power: 1, H2O: -20}, cost=cost_dict['HO']['moderate']['PV']['0'],
             prod_max=100, gwp=53000, land=13320/1800, trl='nrel', block='power_generation', \
                 label='Solar photovoltaics (PV) array', citation='Use pvlib conversion', varying= True)

AKE = Process(name='AKE', intro_scale=ake_start, conversion={Power: -1, H2: 19.474, O2: 763.2, H2O: -175.266},
              cost=cost_dict['HO']['moderate']['AKE']['0'], prod_max=bigM, trl='utility', block='material_production',
              label='Alkaline water electrolysis (AWE)', citation='Demirhan et al. 2018 AIChE paper')  # 20.833 MW required to produce 1000t/day.H2

SMRH = Process(name='SMRH', intro_scale=smrh_start, conversion={Power: -1.11*10**(-3), CH4: -3.76, H2O: -23.7, H2: 1, CO2_Vent: 1.03, CO2: 9.332},
               cost=cost_dict['HO']['moderate']['SMRH']['0'], prod_max=bigM, gwp=0, trl='enterprise', block='material_production',
               label='Steam methane reforming + CCUS', citation='Mosca 2020, 90pc capture')

SMR = Process(name='SMR', intro_scale=smr_start, cost= {'CAPEX': 2400, 'Fixed O&M': 800, 'Variable O&M': 0.03, 'units': '$/kg', 'source': 'dummy'}, \
    conversion={Power: -1.11*10**(-3), CH4: -3.76, H2O: -23.7, H2: 1, CO2_Vent: 9.4979}, prod_max=bigM, gwp=0, trl='enterprise',
                      block='material_production', label='Steam methane reforming', citation='Mosca 2020')

ASMR = Process(name='ASMR', conversion={Uranium: -4.17*10**(-5), H2O: -3364.1, Power: 1}, cost=cost_dict['HO']['moderate']['ASMR']['0'],
               intro_scale=asmr_start, gwp=9100, prod_max=bigM, land=1100/1800, trl='pilot', block='power_generation', label='Small modular reactors (SMRs)')

EOR = Process(name='EOR', intro_scale=0, conversion={Power: -0.00255, CO2: -1, CO2_EOR: 1, CO2_Vent: 0.67},
              cost=cost_dict['HO']['moderate']['EOR']['0'], prod_max=bigM, carbon_credit=True,
              trl='enterprise', block='CCUS', label='CO2-Enhanced oil recovery')

AQoff_SMR = Process(name='AQoff_SMR', conversion={Power: -0.00128, CO2_AQoff: 1, CO2: -1}, cost=cost_dict['HO']['moderate']['AQoff_SMR']['0'],
                    prod_max=bigM, carbon_credit=True, trl='repurposed', block='CCUS', label='Offshore aquifer CO2 sequestration (SMR)')

#TODO
ASU = Process(name = 'ASU', conversion = {Air: -1, N2: 1}, prod_max= bigM, gwp = 0, \
    cost={'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': 1, 'units': '$/kg', 'source': 'dummy'}, label = 'air separation')

HB = Process(name = 'HB', conversion = {H2: -1, N2:-1, NH3: 1}, prod_max= bigM, gwp = 0, \
    cost={'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': 1, 'units': '$/kg', 'source': 'dummy'}, label = 'Haber Bosch')
    

In [38]:
scales = Temporal_scale([1, 365, 24])

In [39]:
processes = {LiI_c, LiI_d, CAES_c, CAES_d, PSH_c, PSH_d, #power storage\ 
    WF, PV, ASMR,  #power production\
        AKE, SMRH, SMR, ASU, HB, #DEC production\
            EOR, AQoff_SMR } # CCU

In [40]:
TX = Location(name='TX', processes= processes, cost_factor = {CH4: ng_price}, \
        capacity_factor = {PV: pandas.DataFrame(weather['dni']), WF: pandas.DataFrame(weather['wind_speed'])},\
            scales=scales, demand_level=1, capacity_level= 2, cost_level= 1, label = 'Texas')

In [43]:
scenario = Scenario(name= 'HyvAmm', network= TX, scales= scales,expenditure_scale_level= 1, \
    scheduling_scale_level = 2, demand_scale_level = 1, network_scale_level = 0, label= 'Hyv Ammonia')

In [44]:
milp = formulate(scenario= scenario, demand= 100,\
        constraints={Constraints.cost, Constraints.inventory, Constraints.production, Constraints.resource_balance}, \
                objective= Objective.cost)

set()


In [ ]:
# milp = formulate(scenario= scenario, \
#     constraints={Constraints.cost, Constraints.inventory, Constraints.production, Constraints.resource_balance}, \
#         objective= Objective.demand)

In [ ]:
# def initial_inventory_rule(instance, location, resource):
#     return  instance.Inv[location, resource, 0] == initial_dict[resource]
# milp.initial_inventory_cons = Constraint(milp.locations, milp.resources_store, rule = initial_inventory_rule )

In [45]:
results = solve(scenario = scenario, instance= milp, solver= 'gurobi', \
    name=f"ammonia_cs", print_solversteps = True)

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 455696 rows, 324356 columns and 1362410 nonzeros
Model fingerprint: 0xab5bff04
Variable types: 324319 continuous, 37 integer (37 binary)
Coefficient statistics:
  Matrix range     [1e-06, 3e+06]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-02, 1e+10]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve removed 267096 rows and 179555 columns
Presolve time: 1.94s
Presolved: 188600 rows, 144801 columns, 565500 nonzeros
Variable types: 144788 continuous, 13 integer (13 binary)
Found heuristic solution: objective 2.394094e+07

Deterministic concurrent LP optimizer: primal and dual simplex
Showing first log only...


Root simplex log...

Iteration    Objective       Primal Inf.    Dual Inf.      Time
   43135    5.9703531e+07   0.000

In [ ]:
milp.X_P.pprint()

X_P : Process Binary
    Size=22, Index=X_P_index
    Key                           : Lower : Value : Upper : Fixed : Stale : Domain
       ('Factory', 'Blender1', 0) :     0 :   1.0 :     1 : False : False : Binary
       ('Factory', 'Blender2', 0) :     0 :   1.0 :     1 : False : False : Binary
    ('Factory', 'CultureSilo', 0) :     0 :   1.0 :     1 : False : False : Binary
      ('Factory', 'CurdTank1', 0) :     0 :   1.0 :     1 : False : False : Binary
      ('Factory', 'CurdTank2', 0) :     0 :   1.0 :     1 : False : False : Binary
      ('Factory', 'CurdTank3', 0) :     0 :   1.0 :     1 : False : False : Binary
        ('Factory', 'Drainer', 0) :     0 :   1.0 :     1 : False : False : Binary
        ('Factory', 'Filler1', 0) :     0 :   1.0 :     1 : False : False : Binary
        ('Factory', 'Filler2', 0) :     0 :   1.0 :     1 : False : False : Binary
    ('Factory', 'LactoseTank', 0) :     0 :   1.0 :     1 : False : False : Binary
       ('Factory', 'MilkSilo', 0) :  

In [ ]:
milp.S.pprint()

S : Resource Dispensed/Sold
    Size=42, Index=S_index
    Key                           : Lower : Value              : Upper : Fixed : Stale : Domain
         ('Factory', 'CH4', 0, 0) :     0 :  892.6464180443396 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 1) :     0 :  892.6464180443419 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 2) :     0 :  892.6464180443419 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 3) :     0 :   892.646418044342 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 4) :     0 :   892.646418044342 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 5) :     0 :   892.646418044342 :  None : False : False : NonNegativeReals
         ('Factory', 'CO2', 0, 0) :     0 :  2059.887222862996 :  None : False : False : NonNegativeReals
         ('Factory', 'CO2', 0, 1) :     0 :  2059.887222863001 :  None : False : False : NonNegativeReals
 

In [ ]:
milp

In [ ]:
# milp.resource_purchase_constraint.pprint()


In [ ]:
# milp.nameplate_production_constraint.pprint()

In [ ]:
# milp.storage_facility_constraint.pprint()

In [ ]:
# milp.location_production_constraint.pprint()

In [ ]:
# milp.inventory_balance_constraint.pprint()


In [ ]:
# milp.demand_constraint.pprint()

In [ ]:
# milp.resources_demand.pprint()


In [ ]:
# milp.resources_sell.pprint()

In [ ]:
# milp.resources_purch.pprint()

In [ ]:
# milp.resources_demand.pprint()

In [ ]:
factory.__dict__.keys()

dict_keys(['name', 'processes', 'scales', 'demand', 'demand_factor', 'cost_factor', 'capacity_factor', 'demand_level', 'cost_level', 'capacity_level', 'label', 'resources', 'materials', 'scale_levels', 'resource_price', 'failure_processes', 'fail_factor'])

In [ ]:
scenario.conversion.keys()

dict_keys(['MilkSilo', 'LactoseTank', 'ProteinTank', 'Vat1', 'Blender2', 'ROMem', 'WasteTank1', 'Warehouse1', 'Filler1', 'Warehouse2', 'Vat3', 'Drainer', 'Vat2', 'CultureSilo', 'CurdTank1', 'CurdTank3', 'WasteTank2', 'Pasteurizer', 'UFMem', 'Filler2', 'Blender1', 'CurdTank2', 'MilkSilo_discharge', 'LactoseTank_discharge', 'ProteinTank_discharge', 'WasteTank1_discharge', 'Warehouse1_discharge', 'Warehouse2_discharge', 'CultureSilo_discharge', 'CurdTank1_discharge', 'CurdTank3_discharge', 'WasteTank2_discharge', 'CurdTank2_discharge'])

In [ ]:
for i in list(scenario.resource_set):
    print(i.name)

StdBld1_stored
Whey
Cream
Waste2
SolCurd
CO2
Permeate
NO2
Waste1_stored
Protein_stored
Curd_stored
StdBld2
PasteurizedMilk
StdBld2_stored
SkimMilk_stored
SkimMilk
StdBld1
Culture_stored
Cheese
Curd
Waste2_stored
Lactose_stored
CH4
Waste1
Protein
Culture
Lactose


In [ ]:
milp.inventory_balance_constraint[factory.name,'StdBld1_stored',0, 1].pprint()

{Member of inventory_balance_constraint} : mass balance across scheduling scale
    Size=162, Index=inventory_balance_constraint_index, Active=True
    Key                                 : Lower : Body                                                                                                                                  : Upper : Active
    ('Factory', 'StdBld1_stored', 0, 1) :   0.0 : P[Factory,Warehouse1,0,1] - P[Factory,Warehouse1_discharge,0,1] - (Inv[Factory,StdBld1_stored,0,1] - Inv[Factory,StdBld1_stored,0,0]) :   0.0 :   True


In [ ]:
milp.storage_facility_constraint.pprint()

storage_facility_constraint : storage facility sizing and location
    Size=9, Index=storage_facility_constraint_index, Active=True
    Key                               : Lower : Body                                                                    : Upper : Active
     ('Factory', 'Culture_stored', 0) :  -Inf :   Cap_S[Factory,Culture_stored,0] - 10000*X_S[Factory,Culture_stored,0] :   0.0 :   True
        ('Factory', 'Curd_stored', 0) :  -Inf :          Cap_S[Factory,Curd_stored,0] - 4000*X_S[Factory,Curd_stored,0] :   0.0 :   True
     ('Factory', 'Lactose_stored', 0) :  -Inf :    Cap_S[Factory,Lactose_stored,0] - 6000*X_S[Factory,Lactose_stored,0] :   0.0 :   True
     ('Factory', 'Protein_stored', 0) :  -Inf :    Cap_S[Factory,Protein_stored,0] - 6000*X_S[Factory,Protein_stored,0] :   0.0 :   True
    ('Factory', 'SkimMilk_stored', 0) :  -Inf : Cap_S[Factory,SkimMilk_stored,0] - 28200*X_S[Factory,SkimMilk_stored,0] :   0.0 :   True
     ('Factory', 'StdBld1_stored', 0) :  -Inf 

In [ ]:
# milp.storage_facility_constraint.pprint()

In [ ]:
milp.demand_constraint.pprint()

: 

: 